In [ ]:
import json
import matplotlib.pyplot as plt
import jax.numpy as jp

def load_metrics(path):
    with open(path, "r") as f:
        return json.load(f)

In [ ]:
proprio_metrics = load_metrics("metrics/proprioceptive_metrics.json")
extero_metrics = load_metrics("metrics/exteroceptive_metrics.json")

policies = {
    "Proprioceptive": proprio_metrics,
    "Exteroceptive": extero_metrics,
}

policy_colors = {
    "Proprioceptive": "tab:blue",
    "Exteroceptive": "tab:orange",
}

fig, axs = plt.subplots(3, 1, figsize=(8, 8))
window = 20

for policy_name, metrics in policies.items():
    for metric in metrics:
        # --- Smooth and plot each metric ---
        smooth_reward = jp.convolve(jp.array(metric["total_reward"]), jp.ones(window)/window, mode='valid')
        axs[0].plot(smooth_reward, label=policy_name, color=colors.get(policy_name))

        smooth_lin_vel = jp.convolve(jp.array(metric["linear_velocity_deviation"]), jp.ones(window)/window, mode='valid')
        axs[1].plot(smooth_lin_vel, label=policy_name, color=colors.get(policy_name))

        smooth_torso = jp.convolve(jp.array(metric["torso_heights"]), jp.ones(window)/window, mode='valid')
        axs[2].plot(smooth_torso, label=policy_name, color=colors.get(policy_name))

        # Get mean and std lin vel error
        mean_error = jp.mean(jp.array(metric["linear_velocity_deviation"]))
        std_error = jp.std(jp.array(metric["linear_velocity_deviation"]))
        print(f"Mean linear velocity tracking error: {mean_error:.3f} ± {std_error:.3f}")

        # Get mean and std torso height
        mean_height = jp.mean(jp.array(metric["torso_heights"]))
        std_height = jp.std(jp.array(metric["torso_heights"]))
        print(f"Torso height mean: {mean_height:.3f}, std: {std_height:.3f}")

axs[0].set_title("Total reward per step")
axs[1].set_title("Linear velocity deviation")
axs[2].set_title("Torso height")

for ax in axs:
    ax.set_xlabel("Timestep")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()